# Fixed-parameter MLP experiment · Colab
Self-contained snapshot of the current `ml_exp` training path; no repository clone or ROCm installation required.

Choose **Runtime → Change runtime type → GPU**, then run the cells in order. Upload the same `data/data.txt` corpus used on your computer and log into W&B when prompted. The default joins your existing fixed-parameter sweep `4oxvbvjf`; it does not create a new sweep. W&B assigns rank, seed, learning rate and step count. `AGENT_RUN_COUNT` limits how many trials this Colab session runs.

For a manual grid, clear `SWEEP_PATH` and edit `RANKS`/`SEEDS`. W&B is optional only in manual mode. The sanity check runs without a corpus.

The fixed-budget MLP includes biases and rounds the hidden width down, so its parameter count is slightly below the full-rank baseline. Results may differ across devices and PyTorch versions. This continues the sweep with new trials; it does not restore an interrupted model or optimizer.


In [ ]:
import importlib.util
import subprocess
import sys

# Keep the runtime's PyTorch installation, including its GPU support.
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])

import json
import random
import time
from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from torch import nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | {device}")
if device == "cuda":
    print(torch.cuda.get_device_name())


## Model and character dataset
Only the classes used by the current experiment are included; their computation matches the source code.

In [ ]:
class Dataset:

    def __init__(self, data_set, train=0.9, device='cpu'):
        print('Dataset loading...')
        self.device = device
        self.data_set = data_set
        self.text = open(data_set, 'r', encoding='utf-8').read()
        self.vocab = sorted(list(set(self.text)))
        self.vocab_size = len(self.vocab)
        self.data = torch.tensor(self.encode(self.text), dtype=torch.long, device=device)
        split_idx = int(train * len(self.data))
        self.train_data = self.data[:split_idx]
        self.val_data = self.data[split_idx:]
        print(f'Dataset loaded from {data_set}.\nVocabulary size: {self.vocab_size}.\nVocabulary sample: {self.vocab[:10]}\nTraining data size: {len(self.train_data)}\nValidation data size: {len(self.val_data)}')

    def encode(self, text):
        return [self.vocab.index(c) for c in text]

    def decode(self, indices):
        return ''.join([self.vocab[i] for i in indices])

    def get_batch(self, type, block_size=16, batch_size=16):
        if type == 'train':
            data = self.train_data
        else:
            data = self.val_data
        offsets = torch.randint(0, len(data) - block_size - 1, (batch_size,))
        x = torch.stack([data[i:i + block_size] for i in offsets])
        y = torch.stack([data[i + 1:i + block_size + 1] for i in offsets])
        return (x, y)

class SineusoidalPositionalEncoding(nn.Module):

    def __init__(self, n_embed, dtype=torch.float32, sinusoidal_base=10000.0, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, n_embed, dtype=dtype)
        self.gamma = nn.Parameter(torch.ones(1, n_embed, dtype=dtype))
        position = torch.arange(0, max_len, dtype=dtype).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, n_embed, 2, dtype=dtype) * (-torch.log(torch.tensor(sinusoidal_base, dtype=dtype)) / n_embed))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.gamma * self.pe[:, :x.size(1), :]

class MultiHeadAttention(nn.Module):

    def __init__(self, head_size, n_embed, n_head, dtype=torch.float32, causal=True, dropout_rate=0.1):
        super().__init__()
        self.head_size = head_size
        self.n_head = n_head
        self.causal = causal
        self.qkv = nn.Linear(n_embed, 3 * n_head * head_size, bias=False, dtype=dtype)
        self.proj = nn.Linear(n_head * head_size, n_embed, dtype=dtype)
        self.dropout = nn.Dropout(dropout_rate)
        self.attention_dropout_rate = dropout_rate

    def forward(self, x, use_cache=None, position_ids=None):
        batch_size, sequence_length, _ = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(batch_size, sequence_length, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(batch_size, sequence_length, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(batch_size, sequence_length, self.n_head, self.head_size).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=self.attention_dropout_rate if self.training else 0.0)
        out = out.transpose(1, 2).reshape(batch_size, sequence_length, -1)
        out = self.dropout(out)
        return self.proj(out)

class MLP_fixed(nn.Module):

    def __init__(self, n_embed, mlp_rank, dtype=torch.float32, dropout_rate=0.1, activation=nn.GELU()):
        super().__init__()
        hidden_layer = int((2 * n_embed ** 2 + n_embed - 2 * mlp_rank * n_embed - 2 * mlp_rank) / (2 * mlp_rank + 1))
        if hidden_layer <= 0:
            raise ValueError('Invalid hidden layer size. Please choose a larger mlp_rank.')
        self.net = nn.Sequential(nn.Linear(n_embed, mlp_rank, dtype=dtype), nn.Linear(mlp_rank, hidden_layer, dtype=dtype), activation, nn.Linear(hidden_layer, mlp_rank, dtype=dtype), nn.Linear(mlp_rank, n_embed, dtype=dtype), nn.Dropout(dropout_rate))

    def forward(self, x):
        return self.net(x)

class FixedSizeTransformerBlock(nn.Module):

    def __init__(self, head_size, n_embed, n_head, mlp_rank, dtype=torch.float32, causal=True, dropout_rate=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(head_size, n_embed, n_head, dtype=dtype, causal=causal, dropout_rate=dropout_rate)
        self.ffwdn = MLP_fixed(n_embed, mlp_rank, dtype=dtype, dropout_rate=dropout_rate, activation=nn.GELU())
        self.layer_norm1 = nn.LayerNorm(n_embed, dtype=dtype)
        self.layer_norm2 = nn.LayerNorm(n_embed, dtype=dtype)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.dropout(x)
        x = x + self.attention(self.layer_norm1(x))
        x = x + self.ffwdn(self.layer_norm2(x))
        return x

class LangModel(nn.Module):

    def __init__(self, vocab_size, block_size, head_size=16, n_embed=32, transformer_blocks=1, n_head=4, mlp_rank=32, dtype=torch.float32, causal=True, dropout_rate=0.1):
        super().__init__()
        self.block_size = block_size
        self.input_n_embed = n_embed
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed, dtype=dtype)
        self.positional_encoding = SineusoidalPositionalEncoding(n_embed, dtype=dtype, max_len=block_size)
        transformer_chain = []
        for _ in range(transformer_blocks):
            transformer_chain.append(FixedSizeTransformerBlock(head_size, self.input_n_embed, n_head=n_head, mlp_rank=mlp_rank, dtype=dtype, causal=causal, dropout_rate=dropout_rate))
            transformer_chain.append(SineusoidalPositionalEncoding(self.input_n_embed, dtype=dtype, max_len=block_size))
        self.transformer_blocks = nn.Sequential(*transformer_chain)
        self.lm_head = nn.Linear(self.input_n_embed, vocab_size, dtype=dtype)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tkn_embs = self.token_embedding_table(idx)
        tkn_embs = self.positional_encoding(tkn_embs)
        ffwdn = self.transformer_blocks(tkn_embs)
        logits = self.lm_head(ffwdn)
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        else:
            loss = None
        return (logits, loss)

    def generation(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


## Quick pipeline check
Checks tokenization, attention heads, MLP dimensions/budget, batches, loss, gradients, an optimizer update, validation and generation on the selected device.

In [ ]:
def check(condition, message):
    if not condition:
        raise RuntimeError(message)

def sanity_check():
    torch.manual_seed(42)
    block_size, batch_size = (16, 2)
    with TemporaryDirectory() as directory:
        source = Path(directory) / 'sample.txt'
        source.write_text('A small language model sanity check.\n' * 100, encoding='utf-8')
        dataset = Dataset(source, device=device)
    check(dataset.decode(dataset.encode(dataset.text)) == dataset.text, 'Token round trip failed')
    model = LangModel(dataset.vocab_size, block_size, head_size=8, n_embed=32, transformer_blocks=2, n_head=4, mlp_rank=4).to(device)
    for block in model.transformer_blocks[::2]:
        check(block.attention.n_head == 4, 'Incorrect attention head count')
        linear_shapes = [(layer.in_features, layer.out_features) for layer in block.ffwdn.net if isinstance(layer, torch.nn.Linear)]
        check(linear_shapes == [(32, 4), (4, 201), (201, 4), (4, 32)], 'Incorrect fixed-budget MLP dimensions')
        check(sum((parameter.numel() for parameter in block.ffwdn.parameters())) == 2105, 'Incorrect MLP parameter count (baseline 2112, rounding gap 7)')
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
    x, y = dataset.get_batch('train', block_size=block_size, batch_size=batch_size)
    check(x.shape == y.shape == (batch_size, block_size), 'Incorrect batch shape')
    check(torch.equal(x[:, 1:], y[:, :-1]), 'Targets are not shifted by one token')
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(x, y)
    check(logits.shape == (batch_size, block_size, dataset.vocab_size), 'Incorrect logits shape')
    check(torch.isfinite(logits).all().item(), 'Non-finite logits')
    check(loss is not None and loss.ndim == 0 and torch.isfinite(loss).item(), 'Invalid loss')
    loss.backward()
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            check(parameter.grad is not None, f'Missing gradient: {name}')
            check(torch.isfinite(parameter.grad).all().item(), f'Non-finite gradient: {name}')
    before = model.lm_head.weight.detach().clone()
    optimizer.step()
    check(not torch.equal(before, model.lm_head.weight), 'Optimizer did not update weights')
    for name, parameter in model.named_parameters():
        check(torch.isfinite(parameter).all().item(), f'Non-finite parameter after update: {name}')
    model.eval()
    with torch.no_grad():
        x_val, y_val = dataset.get_batch('val', block_size=block_size, batch_size=batch_size)
        _, val_loss = model(x_val, y_val)
        check(torch.isfinite(val_loss).item(), 'Non-finite validation loss')
        generated = model.generation(x_val[:1], max_new_tokens=4)
        check(generated.shape == (1, block_size + 4), 'Incorrect generation shape')
        dataset.decode(generated[0].tolist())
    print(f'PASS on {device}: forward, backward, optimizer, validation, generation (train loss={loss.item():.4f}, val loss={val_loss.item():.4f})')

sanity_check()


## Run settings
The latest local run records entity `boonsvj-university-of-manchester-students-union`, project `ml-exp`, group `mlp-fixed-param-count-sweep`, and sweep ID `4oxvbvjf`. The sweep configuration name is also `mlp-fixed-param-count-sweep`; its objective is to minimize `best_val_loss`.

In sweep mode, `RANKS` and `SEEDS` are ignored: the existing W&B controller assigns trials. Use the original corpus and model settings to keep results comparable. No `wandb.sweep()` call is made.

For a manual grid, set `SWEEP_PATH = ""`. The full local grid is `RANKS = [1, 2, 4, 8, 16, 32, 64]` and `SEEDS = [1337, 1338, 1339, 1340, 1341]`. Set `USE_WANDB = False` for local-only results; `DATA_PATH = "sample"` exercises toy training in manual mode.


In [ ]:
DATA_PATH = "data.txt"
RANKS = [1]
SEEDS = [1337]
USE_WANDB = True
# Join the same sweep as the latest local run; leave empty for a manual grid.
WANDB_ENTITY = "boonsvj-university-of-manchester-students-union"
WANDB_PROJECT = "ml-exp"
WANDB_GROUP = "mlp-fixed-param-count-sweep"
SWEEP_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}/4oxvbvjf"
AGENT_RUN_COUNT = None  # None: keep requesting runs; set 1 for one trial.

CONFIG = dict(
    block_size=128, batch_size=16, head_size=8, n_embed=128, n_head=4,
    transformer_blocks=6, dropout_rate=0.2, causal=True, train_fraction=0.9,
    learning_rate=0.0006, step_count=10_000, loss_est_count=100,
    checkpoint_interval=100, pretrain_generation_tokens=50,
    posttrain_generation_tokens=500,
    eval_prompts=["Once upon a time ", "Klein Moretti was ", "Melissa was a "],
)

if SWEEP_PATH and not USE_WANDB:
    raise ValueError("An existing W&B sweep requires USE_WANDB=True; clear SWEEP_PATH for local runs.")
if SWEEP_PATH and DATA_PATH == "sample":
    raise ValueError("Use the original corpus for the existing sweep; clear SWEEP_PATH for a toy run.")

if USE_WANDB:
    if importlib.util.find_spec("wandb") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb"])
    import wandb
    wandb.login()

if DATA_PATH == "sample":
    source = Path("sample.txt")
    source.write_text(" ".join(CONFIG["eval_prompts"]) * 200, encoding="utf-8")
else:
    source = Path(DATA_PATH)
    if not source.exists():
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one UTF-8 text corpus, then rerun this cell.")
        source = Path(next(iter(uploaded)))
        DATA_PATH = str(source)

dataset = Dataset(source, train=CONFIG["train_fraction"], device=device)
if min(len(dataset.train_data), len(dataset.val_data)) <= CONFIG["block_size"] + 1:
    raise ValueError("Corpus too short for the chosen block size and split.")
for prompt in CONFIG["eval_prompts"]:
    if set(prompt) - set(dataset.vocab):
        raise ValueError(f"Prompt contains characters absent from the corpus: {prompt!r}. Edit eval_prompts.")
for key in ("step_count", "loss_est_count", "checkpoint_interval"):
    if CONFIG[key] < 1:
        raise ValueError(f"{key} must be positive.")


## Train and evaluate
Each rank/seed pair gets its own folder with configuration, loss history, summary and pre/post-training text. As in the script, evaluation runs after the first optimizer step, every checkpoint interval, and at the final step. The notebook also saves a final model checkpoint.
In sweep mode this cell starts an agent for the existing sweep. Its Python callback replaces the desktop-specific command in the sweep YAML. See the [W&B agent documentation](https://docs.wandb.ai/models/ref/python/functions/agent).


In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)


@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.empty(cfg["loss_est_count"], device=device)
        for k in range(len(losses)):
            x, y = dataset.get_batch(split, cfg["block_size"], cfg["batch_size"])
            _, losses[k] = model(x, y)
        out[split] = losses.mean().item()
    model.train()
    return out


@torch.no_grad()
def generate_outputs(model, count, cfg):
    model.eval()
    outputs = []
    for prompt in cfg["eval_prompts"]:
        tokens = torch.tensor([dataset.encode(prompt)], dtype=torch.long, device=device)
        text = dataset.decode(model.generation(tokens, count)[0].tolist())
        outputs.append({"prompt": prompt, "generation": text})
        print(f"Prompt: {prompt}\n{text}")
    return outputs


def train_run(rank, seed, cfg=None, run=None):
    cfg = dict(CONFIG if cfg is None else cfg)
    owns_run = run is None
    set_seed(seed)
    model = LangModel(
        dataset.vocab_size, cfg["block_size"], head_size=cfg["head_size"],
        n_embed=cfg["n_embed"], transformer_blocks=cfg["transformer_blocks"],
        n_head=cfg["n_head"], mlp_rank=rank, dtype=torch.float32,
        causal=cfg["causal"], dropout_rate=cfg["dropout_rate"],
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"])
    n = cfg["n_embed"]
    config = dict(cfg, mlp_rank=rank, seed=seed, device=device, dtype=str(torch.float32),
                  data_source=str(source), vocab_size=dataset.vocab_size,
                  hidden_layer_size=int((2*n*n + n - 2*rank*n - 2*rank)/(2*rank + 1)),
                  parameter_count=sum(p.numel() for p in model.parameters()),
                  optimizer="AdamW", torch_version=str(torch.__version__))
    folder = Path("colab_runs") / f"{datetime.now():%Y-%m-%d_%H-%M-%S-%f}_rank-{rank}_seed-{seed}"
    folder.mkdir(parents=True, exist_ok=False)
    (folder / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    if run is None and USE_WANDB:
        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, config=config)
    if run:
        run.config.update(config, allow_val_change=True)
    try:
        print(f"Rank {rank}, seed {seed} | {config['parameter_count']:,} parameters | {folder}")
        pre = generate_outputs(model, cfg["pretrain_generation_tokens"], cfg)
        if run:
            run.log({"pretrain_generations": wandb.Table(columns=["prompt", "generation"],
                     data=[[o["prompt"], o["generation"]] for o in pre])})
        model.train()
        history = []
        best_train = best_val = float("inf")
        best_step = 0
        start = interval_start = time.perf_counter()
        previous_steps = 0
        for step in range(cfg["step_count"]):
            x, y = dataset.get_batch("train", cfg["block_size"], cfg["batch_size"])
            _, loss = model(x, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            if step % cfg["checkpoint_interval"] == 0 or step == cfg["step_count"] - 1:
                if device == "cuda":
                    torch.cuda.synchronize()
                completed = step + 1
                ms = (time.perf_counter() - interval_start) * 1000 / (completed - previous_steps)
                losses = estimate_loss(model, cfg)
                best_train = min(best_train, losses["train"])
                if losses["val"] < best_val:
                    best_val, best_step = losses["val"], step
                end = time.perf_counter()
                elapsed = end - start
                eta = elapsed / completed * (cfg["step_count"] - completed)
                metrics = dict(train_loss=losses["train"], val_loss=losses["val"],
                               milliseconds_per_step=ms, elapsed_seconds=elapsed, eta_seconds=eta)
                history.append(dict(step=step, **metrics))
                (folder / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
                print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}, "
                      f"{ms:.2f} ms/step, elapsed {elapsed:.0f}s, ETA {eta:.0f}s")
                if run:
                    run.log(metrics, step=step)
                interval_start, previous_steps = end, completed
        post = generate_outputs(model, cfg["posttrain_generation_tokens"], cfg)
        summary = dict(mlp_rank=rank, seed=seed, best_val_loss=best_val,
                       best_train_loss=best_train, best_step=best_step,
                       final_train_loss=losses["train"], final_val_loss=losses["val"])
        (folder / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        (folder / "generations.json").write_text(json.dumps(dict(pretrain=pre, posttrain=post), indent=2), encoding="utf-8")
        torch.save(dict(model_state_dict=model.state_dict(), config=config, vocab=dataset.vocab),
                   folder / "final_model.pt")
        if run:
            run.log({"posttrain_generations": wandb.Table(columns=["prompt", "generation"],
                     data=[[o["prompt"], o["generation"]] for o in post])})
            run.summary.update(summary)
        return summary
    finally:
        if run and owns_run:
            run.finish()


summaries = []


def sweep_trial():
    # W&B supplies the next trial from the existing sweep, overriding defaults.
    with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                    group=WANDB_GROUP, config=CONFIG) as run:
        cfg = dict(CONFIG)
        cfg.update(dict(run.config))
        summaries.append(train_run(cfg["mlp_rank"], cfg["seed"], cfg=cfg, run=run))


if SWEEP_PATH:
    wandb.agent(SWEEP_PATH, function=sweep_trial, count=AGENT_RUN_COUNT)
else:
    summaries = [train_run(rank, seed) for rank in RANKS for seed in SEEDS]
print(json.dumps(summaries, indent=2))


## Download results
Run this cell to download all run folders, including final checkpoints. Colab runtime files are temporary; download results before disconnecting.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("colab_runs", "zip", "colab_runs")
files.download(archive)
